In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # UTC+8 (kept for consistency; timestamps left as-is)

# Load CSV files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Ensure value columns are numeric where applicable
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# 1) metric.csv aggregates: per (cmdb_id, kpi_name) compute global P95 and count
metric_summary = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
)
metric_summary = metric_summary[['cmdb_id', 'kpi_name', 'count', 'p95_value']].head(20)

# 2) trace.csv aggregates: per (cmdb_id, trace_name) compute global P95 and count
trace_summary = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
)
trace_summary = trace_summary[['cmdb_id', 'trace_name', 'count', 'p95_value']].head(20)

# 3) log.csv aggregates: per (cmdb_id, log_name) compute global P95 and count
log_summary = (
    log.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
)
log_summary = log_summary[['cmdb_id', 'log_name', 'count', 'p95_value']].head(20)

# 4) error_logs.csv: count per cmdb_id (sorted desc) and up to 5 earliest timestamped rows overall
error_counts = (
    error_logs.groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

error_earliest = error_logs.sort_values('timestamp', ascending=True)[['timestamp', 'cmdb_id', 'message']].head(5)

# Display the compact summaries (use variable names as outputs)
metric_summary, trace_summary, log_summary, error_counts, error_earliest

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregates:\n\n"
    "1) Metrics (top by point count): Most (cmdb_id, kpi_name) series have 31 points. "
    "Notable P95s include IG02 OSLinux-CPU_CPU_CPUUserTime ≈ 25.33, IG01 JVM_Memory_HeapMemoryUsed ≈ 8.15e8, "
    "IG01 JVM_ThreadCount P95 ≈ 74.5, and Tomcat04 ErrorCount P95 ≈ 274.0. Several OS and JVM KPIs appear in the top-20 by count.\n\n"
    "2) Traces (top by point count): IG01 and IG02 dominate (30 points each). "
    "trace.self.duration_mean P95 is ~9.97 (IG01) and ~10.91 (IG02); trace.self.duration_p95 is ~39.19 for both. "
    "Inter-service call p95s to Tomcat nodes are around 2.07–2.21s; trace row_count P95s are ~1009–1048.\n\n"
    "3) Logs (top by point count): Many log series have 31 points. apache02 and several Tomcat nodes show log.row_count P95s in the hundreds (e.g., apache01 ~513.5, apache02 ~509.5, Tomcat01 ~260.5). "
    "log.error_count P95s are 0 for the top entries (no high aggregated error-rate shown by P95).\n\n"
    "4) Error logs: apache02 has the most error_log records (6), apache01 has 3. "
    "The five earliest error-log entries occur at timestamps 1615288704–1615288706 and are HTTP POST-related messages from apache02 and apache01.\n\n"
    "Overall: CPU, JVM memory/threads, and Tomcat request/error metrics appear frequently in the top series by volume; traces show relatively high tail latency (self.duration_p95 ~39s) and per-call p95s to Tomcat ~2s. Log error counts are zero at the P95 level, while error_logs are concentrated on apache02 and apache01 (few records)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     p95_value
102       IG02                        OSLinux-CPU_CPU_CPUUserTime     31  2.532941e+01
101       IG02                         OSLinux-CPU_CPU_CPUSysTime     31  4.488000e-01
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.450000e+01
1586  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  5.000000e-01
100       IG02                            OSLinux-CPU_CPU_CPULoad     31  1.175000e+00
104       IG02                        OSLinux-CPU_CPU_CPUidleutil     31  7.423985e+01
1659  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00
1603  apache02                            OSLinux-CPU_CPU_CPULoad     31  9.500000e-02
1607  apache02                        OSLinux-CPU_CPU_CPUidleutil     31  9.963843e+01
1662  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00
1606  apache02                             OSLinux-CPU_CPU_CPUWio     31  3.907000e-01
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.794029e+07
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  8.151600e+08
1605  apache02                        OSLinux-CPU_CPU_CPUUserTime     31  1.890665e-01
87        IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.488325e+06
1495  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  2.740000e+02
1503  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00
1502  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00
1511  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.670000e+02
88        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09,    cmdb_id                       trace_name  count    p95_value
0     IG01         trace.self.duration_mean     30     9.966838
1     IG01          trace.self.duration_p95     30    39.186712
2     IG01             trace.self.row_count     30  1009.000000
3     IG01  trace.to_Tomcat01.duration_mean     30     0.404355
4     IG01   trace.to_Tomcat01.duration_p95     30     2.094550
5     IG01      trace.to_Tomcat01.row_count     30   128.300000
6     IG01  trace.to_Tomcat02.duration_mean     30     0.423491
7     IG01   trace.to_Tomcat02.duration_p95     30     2.082262
8     IG01      trace.to_Tomcat02.row_count     30   128.950000
9     IG01  trace.to_Tomcat03.duration_mean     30     0.415120
10    IG01   trace.to_Tomcat03.duration_p95     30     2.074715
11    IG01      trace.to_Tomcat03.row_count     30   130.100000
12    IG01  trace.to_Tomcat04.duration_mean     30     0.464100
13    IG01   trace.to_Tomcat04.duration_p95     30     2.108075
14    IG01      trace.to_Tomcat04.row_count     30   132.100000
15    IG02         trace.self.duration_mean     30    10.906891
16    IG02          trace.self.duration_p95     30    39.184513
17    IG02             trace.self.row_count     30  1048.400000
18    IG02  trace.to_Tomcat01.duration_mean     30     0.377095
19    IG02   trace.to_Tomcat01.duration_p95     30     2.211130,      cmdb_id          log_name  count  p95_value
18  apache02   log.error_count     31      0.000
10  Tomcat03   log.error_count     31      0.000
6   Tomcat01     log.row_count     31    260.500
4   Tomcat01   log.error_count     31      0.000
19  apache02     log.row_count     31    509.500
17  apache01     log.row_count     31    513.500
13  Tomcat04   log.error_count     31      0.000
12  Tomcat03     log.row_count     31    260.000
16  apache01   log.error_count     31      0.000
15  Tomcat04     log.row_count     31    268.500
7   Tomcat02   log.error_count     18      0.000
9   Tomcat02     log.row_count     18      1.000
8   Tomcat02  log.gc_pause_max     18      0.110
14  Tomcat04  log.gc_pause_max     12      0.109
11  Tomcat03  log.gc_pause_max     11      0.085
5   Tomcat01  log.gc_pause_max     11      0.085
3    Mysql02     log.row_count      1     68.000
2    Mysql02   log.error_count      1      0.000
1    Mysql01     log.row_count      1      9.000
0    Mysql01   log.error_count      1      0.000,     cmdb_id  count
1  apache02      6
0  apache01      3,     timestamp   cmdb_id                                            message
0  1615288704  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
1  1615288704  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
2  1615288705  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
3  1615288705  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
4  1615288706  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import datetime

# reuse existing 'metric' DataFrame loaded earlier in the kernel
# ensure numeric types
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# timezone and incident window in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 0, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 1) Compute global P95 per (cmdb_id, kpi_name) across the entire file BEFORE any filtering
p95_df = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Filter metric to the incident window (timestamps left as-is)
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 3) Join global p95 thresholds into the windowed data
mw = metric_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# If there is no data in window, produce explicit message
if mw.shape[0] == 0:
    anomalies_summary = "No metric points in the specified incident window ({} to {}).".format(start_ts, end_ts)
else:
    # detect anomalies where value >= global_p95 (threshold computed globally)
    mw['is_anom'] = mw['value'] >= mw['global_p95']

    # aggregate totals and maxima per series in the window
    full_agg = (
        mw.groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(total_points_in_window=('value', 'count'),
             max_value_in_window=('value', 'max'),
             global_p95=('global_p95', 'first'))
        .reset_index()
    )

    # aggregate anomalies (only for anomaly rows)
    anom_agg = (
        mw[mw['is_anom']]
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(anomaly_count=('is_anom', 'sum'),
             earliest_anomaly_timestamp=('timestamp', 'min'))
        .reset_index()
    )

    # merge and fill zeros for non-anomalous series
    summary = full_agg.merge(anom_agg, on=['cmdb_id', 'kpi_name'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
    # keep earliest_anomaly_timestamp as-is (integers) or NA if none
    summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].where(summary['earliest_anomaly_timestamp'].notna(), pd.NA)

    # filter to series that have at least one anomaly
    anomalies_summary = summary[summary['anomaly_count'] > 0].copy()

    if anomalies_summary.shape[0] == 0:
        anomalies_summary = "No anomalies found in the window ({} to {}) using global P95 thresholds.".format(start_ts, end_ts)
    else:
        # sort by anomaly_count desc and limit to top 20 rows
        anomalies_summary = (
            anomalies_summary.sort_values('anomaly_count', ascending=False)
            [['cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count',
              'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95']]
            .head(20)
            .reset_index(drop=True)
        )

# display the compact result variable
anomalies_summary

```
Out[2]:
```
Summary of anomaly detection (metrics) for the incident window (March 9, 2021 19:00–19:30 UTC+8):

- Anomalies were found. The top series (up to 20 shown) all have anomaly_count = 31 and total_points_in_window = 31, meaning every sample in the 31-minute window met the condition value >= global P95. The earliest anomaly timestamp reported for all listed series is 1615287600.

- Affected components and metric types (examples from the top results):
  - IG01: JVM metrics (JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed, JVM-Memory_7778_JVM_Memory_HeapMemoryMax) — global_p95 equals observed max (5.794029e+07 and 2.040136e+09 respectively), anomaly_count 31.
  - apache02 / apache01: various OSLinux process and network metrics (process PROCPPCount, TCP CLOSE-WAIT/FIN-WAIT counters, apache process counts) — many have global_p95 equal to the observed values, resulting in anomaly_count 31.
  - Redis01/Redis02, Mysql01/Mysql02: OSLinux network/process metrics (TCP CLOSE-WAIT/FIN-WAIT and PROCNoZombies/PROCPPCount) — similarly flagged across the window.

- Important interpretation note: many flagged series have global P95 equal to zero or small integer counts (e.g., TCP CLOSE-WAIT = 0, small PROCPPCount values). Because the detection condition was value >= global P95, equality to zero or to small integers produces many trivial matches. In other words, several "anomalies" reflect value == P95 (including zeros), not necessarily large deviations.

- Takeaway: Multiple series across IG01, apache02/apache01, Redis and Mysql nodes are flagged for the entire incident window. The most notable non-trivial items to review are the IG01 JVM memory metrics (where P95 equals the observed high values). For other flagged series, verify whether equality-to-zero or small-count equality is meaningful before treating them as true degradations.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_p95
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                      31             31                  1615287600         5.794029e+07  5.794029e+07
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                      31             31                  1615287600         2.040136e+09  2.040136e+09
2   apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
3   apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31             31                  1615287600         7.000000e+00  7.000000e+00
4   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31             31                  1615287600         0.000000e+00  0.000000e+00
5   apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount                      31             31                  1615287600         6.000000e+00  6.000000e+00
6    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31             31                  1615287600         0.000000e+00  0.000000e+00
7    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
8    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
9       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
10  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                      31             31                  1615287600         5.000000e-01  5.000000e-01
11      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31             31                  1615287600         3.000000e+00  3.000000e+00
12      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31             31                  1615287600         1.000000e+00  1.000000e+00
13   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31             31                  1615287600         4.000000e+00  4.000000e+00
14   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31             31                  1615287600         1.000000e+00  1.000000e+00
15   Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31             31                  1615287600         0.000000e+00  0.000000e+00
16   Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
17   Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
18   Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31             31                  1615287600         0.000000e+00  0.000000e+00
19   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31             31                  1615287600         4.000000e+00  4.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import datetime

# reuse existing 'trace' DataFrame loaded earlier; ensure numeric
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# timezone and incident window in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 0, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 1) Compute global P95 per (cmdb_id, trace_name) across the entire file BEFORE filtering
trace_p95 = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Filter trace to the incident window (timestamps left as-is)
trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# 3) Merge global p95 into windowed data
tw = trace_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# Handle case of no data in window
if tw.shape[0] == 0:
    anomalies_summary = "No trace points in the specified incident window ({} to {}).".format(start_ts, end_ts)
else:
    # detect anomalies where value >= global_p95 (global threshold computed before filtering)
    tw['is_anom'] = tw['value'] >= tw['global_p95']

    # aggregate totals and maxima per series in the window
    full_agg = (
        tw.groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(total_points_in_window=('value', 'count'),
             max_value_in_window=('value', 'max'),
             global_p95=('global_p95', 'first'))
        .reset_index()
    )

    # aggregate anomalies
    anom_agg = (
        tw[tw['is_anom']]
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(anomaly_count=('is_anom', 'sum'),
             earliest_anomaly_timestamp=('timestamp', 'min'))
        .reset_index()
    )

    # merge and fill zeros for non-anomalous series
    summary = full_agg.merge(anom_agg, on=['cmdb_id', 'trace_name'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
    summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].where(summary['earliest_anomaly_timestamp'].notna(), pd.NA)

    # filter to series that have at least one anomaly
    anomalies_summary = summary[summary['anomaly_count'] > 0].copy()

    if anomalies_summary.shape[0] == 0:
        anomalies_summary = "No anomalies found in the window ({} to {}) using global P95 thresholds.".format(start_ts, end_ts)
    else:
        # sort by anomaly_count desc and limit to top 20 rows; keep specified columns
        anomalies_summary = (
            anomalies_summary.sort_values('anomaly_count', ascending=False)
            [['cmdb_id', 'trace_name', 'total_points_in_window', 'anomaly_count',
              'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95']]
            .head(20)
            .reset_index(drop=True)
        )

# display compact result
anomalies_summary

```
Out[3]:
```
summary = (
    "Trace anomaly detection summary for March 9, 2021 19:00–19:30 (UTC+8):\n\n"
    "• Found 20 trace series with anomaly points (top-20 by anomaly count shown).\n\n"
    "Highlights:\n"
    "- dockerB2 / trace.self.duration_p95: 30 points in window, 5 anomalies, earliest ts 1615287600, max=0.0104 vs global_p95=0.0100 (small exceedance).\n"
    "- MG02: multiple row_count series flagged (from_dockerB1, from_Tomcat02, from_dockerA2) — each 30 points, 3 anomalies; e.g. from_dockerB1 max=174 vs p95=152.\n"
    "- Tomcat01 / trace.self.row_count: 30 points, 3 anomalies, earliest ts 1615287600, max=1785 vs p95=1737 (notable increase in row_count).\n"
    "- Several Tomcat/IG traces show 2–3 anomaly points: IG01→Tomcat03 (duration_mean, duration_p95, row_count), IG01→Tomcat04 (duration_p95,row_count); IG02 self.duration_mean/p95 and to_Tomcat01 p95 also show 2 anomaly points each. Example: IG02 trace.self.duration_p95 max=39.2079 vs p95=39.1845 (very slight exceedance).\n\n"
    "Interpretation / notes:\n"
    "- Many detections are small exceedances of the global P95 or equality to integer counts (e.g., row_count or TCP counters equal to their P95). Equality to P95 (including zeros) will be flagged by the >= rule and may be low-severity or expected behavior.\n"
    "- The most actionable items to investigate are series with meaningful increases above P95 (e.g., Tomcat01 row_count, MG02 from_dockerB1 row_count, and IG1/IG2 trace latency p95/mean where values exceed the global P95).\n\n"
    "Timestamps are shown as raw epoch seconds (UTC+8)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window   global_p95
0   dockerB2          trace.self.duration_p95                      30              5                  1615287600             0.010400     0.010000
1       MG02    trace.from_dockerB1.row_count                      30              3                  1615288020           174.000000   152.000000
2   Tomcat01             trace.self.row_count                      30              3                  1615287600          1785.000000  1737.000000
3       MG02    trace.from_Tomcat02.row_count                      30              3                  1615288740           140.000000   140.000000
4       MG02    trace.from_dockerA2.row_count                      30              3                  1615288740            30.000000    30.000000
5   Tomcat03          trace.to_MG01.row_count                      30              3                  1615287660           136.000000   129.000000
6   dockerA2          trace.to_MG02.row_count                      30              3                  1615288740            30.000000    30.000000
7   Tomcat02          trace.to_MG02.row_count                      30              3                  1615288740           140.000000   140.000000
8       MG01    trace.from_Tomcat03.row_count                      30              3                  1615287660           136.000000   129.000000
9   dockerB1          trace.to_MG02.row_count                      30              3                  1615288020           174.000000   152.000000
10      IG01  trace.to_Tomcat03.duration_mean                      30              2                  1615288020             0.504350     0.415120
11      IG01      trace.to_Tomcat03.row_count                      30              2                  1615288920           133.000000   130.100000
12      IG01   trace.to_Tomcat03.duration_p95                      30              2                  1615288020             2.114500     2.074715
13      IG01   trace.to_Tomcat04.duration_p95                      30              2                  1615288620             2.163900     2.108075
14      IG01      trace.to_Tomcat04.row_count                      30              2                  1615287720           138.000000   132.100000
15      IG02         trace.self.duration_mean                      30              2                  1615288500            11.725441    10.906891
16      IG02          trace.self.duration_p95                      30              2                  1615288560            39.207900    39.184513
17      IG02             trace.self.row_count                      30              2                  1615287720          1072.000000  1048.400000
18      IG02  trace.to_Tomcat01.duration_mean                      30              2                  1615288380             0.394510     0.377095
19      IG02   trace.to_Tomcat01.duration_p95                      30              2                  1615288380             2.615250     2.211130```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import datetime

# reuse existing DataFrames: log, error_logs (loaded earlier)

# Components of interest
cmdb_list = ['Tomcat01', 'Tomcat04', 'apache01', 'apache02', 'IG01', 'IG02']

# Incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 0, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 1) Filter log.csv to window and components, sort ascending, select columns, limit 20 rows
log_window = log[
    (log['timestamp'] >= start_ts) & 
    (log['timestamp'] <= end_ts) & 
    (log['cmdb_id'].isin(cmdb_list))
].sort_values('timestamp', ascending=True)

log_window_head = log_window[['timestamp', 'cmdb_id', 'log_name', 'value']].head(20)

# which components have no log rows in the window
present_logs = log_window['cmdb_id'].unique().tolist()
missing_in_logs = [c for c in cmdb_list if c not in present_logs]

# 2) Filter error_logs.csv to same window and components, sort asc, select columns, limit 20 rows
error_window = error_logs[
    (error_logs['timestamp'] >= start_ts) &
    (error_logs['timestamp'] <= end_ts) &
    (error_logs['cmdb_id'].isin(cmdb_list))
].sort_values('timestamp', ascending=True)

error_window_head = error_window[['timestamp', 'cmdb_id', 'message']].head(20)

# which components have no error_log rows in the window
present_errors = error_window['cmdb_id'].unique().tolist()
missing_in_error_logs = [c for c in cmdb_list if c not in present_errors]

# Display compact results
log_window_head, error_window_head, missing_in_logs, missing_in_error_logs

```
Out[4]:
```
summary = (
    "Log and error_log results for March 9, 2021 19:00:00–19:30:00 (UTC+8):\n\n"
    "Logs (filtered to Tomcat01, Tomcat04, apache01, apache02, IG01, IG02):\n"
    "- Returned up to 20 rows (sorted by timestamp ascending) for the window. Present components: Tomcat01, Tomcat04, apache01, apache02.\n"
    "- Example/earliest rows at timestamp 1615287600:\n"
    "  * Tomcat01: log.error_count=0.00, log.gc_pause_max=0.05, log.row_count=262.00\n"
    "  * Tomcat04: log.error_count=0.00, log.gc_pause_max=0.06, log.row_count=221.00\n"
    "  * apache01: log.error_count=0.00, log.row_count=492.00\n"
    "  * apache02: log.error_count=0.00, log.row_count=494.00\n"
    "- Several subsequent timestamps show similar rows (row_count and occasional gc_pause_max). \n"
    "- IG01 and IG02 had no log rows in the incident window.\n\n"
    "Error logs (same window):\n"
    "- Returned up to 20 rows (sorted by timestamp ascending). Only apache01 and apache02 have error_log entries in the window.\n"
    "- Earliest error_log entries are at timestamps starting 1615288704 (apache02 and apache01) containing HTTP POST request messages.\n"
    "- Tomcat01, Tomcat04, IG01, and IG02 have no error_log rows in the window.\n\n"
    "Summary conclusion: application and webserver log metrics (row_count, error_count, small gc_pause values) are present for Tomcat01/Tomcat04/apache01/apache02 during the window; IG01/IG02 produced no log lines. Error log messages during the window are limited to apache01/apache02 (HTTP POST messages)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id          log_name   value
4   1615287600  Tomcat01   log.error_count    0.00
5   1615287600  Tomcat01  log.gc_pause_max    0.05
6   1615287600  Tomcat01     log.row_count  262.00
9   1615287600  Tomcat04   log.error_count    0.00
10  1615287600  Tomcat04  log.gc_pause_max    0.06
11  1615287600  Tomcat04     log.row_count  221.00
12  1615287600  apache01   log.error_count    0.00
13  1615287600  apache01     log.row_count  492.00
14  1615287600  apache02   log.error_count    0.00
15  1615287600  apache02     log.row_count  494.00
16  1615287660  Tomcat01   log.error_count    0.00
17  1615287660  Tomcat01     log.row_count  200.00
24  1615287660  Tomcat04   log.error_count    0.00
25  1615287660  Tomcat04     log.row_count  230.00
26  1615287660  apache01   log.error_count    0.00
28  1615287660  apache02   log.error_count    0.00
29  1615287660  apache02     log.row_count  439.00
27  1615287660  apache01     log.row_count  454.00
30  1615287720  Tomcat01   log.error_count    0.00
38  1615287720  Tomcat04  log.gc_pause_max    0.08,     timestamp   cmdb_id                                            message
0  1615288704  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
1  1615288704  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
2  1615288705  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
3  1615288705  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
4  1615288706  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...
5  1615288706  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
6  1615288706  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso...
7  1615288707  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
8  1615288708  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json..., ['IG01', 'IG02'], ['Tomcat01', 'Tomcat04', 'IG01', 'IG02'])```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import datetime

# Reuse existing 'metric' DataFrame; ensure numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Timezone and incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 0, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Identify IG01 JVM-related KPIs (contain 'JVM-Memory' or 'JVM-Threads')
ig01_mask = (metric['cmdb_id'] == 'IG01')
jvm_mask = metric['kpi_name'].str.contains(r'JVM-Memory|JVM-Threads', na=False)
ig01_jvm_kpis = metric[ig01_mask & jvm_mask]['kpi_name'].unique()

# Compute global P95 per kpi_name for IG01 (across full file, before filtering)
p95_df = (
    metric[ig01_mask & jvm_mask]
    .groupby('kpi_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Filter metric to the incident window and to IG01
metric_window_ig01 = metric[
    (metric['timestamp'] >= start_ts) &
    (metric['timestamp'] <= end_ts) &
    (metric['cmdb_id'] == 'IG01')
].copy()

# Merge global p95 into the windowed IG01 rows (by kpi_name)
mw = metric_window_ig01.merge(p95_df, on='kpi_name', how='left')

# Select rows where value >= that KPI's global_p95 and kpi is JVM-related
if mw.shape[0] == 0:
    ig01_anoms = "No IG01 metric points in the incident window ({} to {}).".format(start_ts, end_ts)
else:
    mw['is_anom'] = mw['value'] >= mw['global_p95']
    ig01_anoms_df = mw[mw['is_anom'] & mw['kpi_name'].isin(ig01_jvm_kpis)].copy()
    if ig01_anoms_df.shape[0] == 0:
        ig01_anoms = "No IG01 JVM-related anomalies found in the window ({} to {}) using global P95 thresholds.".format(start_ts, end_ts)
    else:
        ig01_anoms = (
            ig01_anoms_df[['timestamp', 'cmdb_id', 'kpi_name', 'value', 'global_p95']]
            .sort_values('timestamp', ascending=True)
            .head(20)
            .reset_index(drop=True)
        )

# Display result (compact)
ig01_anoms

```
Out[5]:
```
summary = (
    "IG01 JVM KPI P95 check (global P95 computed across full file) for March 9, 2021 19:00–19:30 (UTC+8):\n\n"
    "- JVM-related KPIs found: JVM-Memory_HeapMemoryMax, JVM-Memory_HeapMemoryUsage, JVM-Memory_NoHeapMemoryUsed (all for cmdb_id IG01).\n"
    "- Anomalies were detected in the incident window (rows where value >= global P95). Earliest anomaly timestamp shown: 1615287600 (epoch seconds, UTC+8).\n"
    "- HeapMemoryMax: value = 2.040136e+09 equals its global P95 repeatedly across the window (many timestamps), i.e., consistently at the P95 level.\n"
    "- NoHeapMemoryUsed: value = 5.794029e+07 equals its global P95 repeatedly across the window as well.\n"
    "- HeapMemoryUsage: there are timestamps where value (e.g., 41.89010) exceeds the global P95 (≈39.67690); at least one timestamp shows equality to the P95 (39.67690).\n\n"
    "Interpretation / recommended next steps:\n"
    "- IG01 shows sustained high heap-related metrics: HeapMemoryMax and NoHeapMemoryUsed are at their global P95 baseline for the incident window, and HeapMemoryUsage exceeded the P95 at least once. This suggests persistent high memory footprint and at least transient higher heap utilization.\n"
    "- Investigate IG01 JVM memory usage (GC behavior, heap sizing, recent deployments or traffic spikes) to determine whether the sustained P95-level values are expected or indicate growing memory pressure that could lead to problems."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp cmdb_id                                     kpi_name         value    global_p95
0   1615287600    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
1   1615287600    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.189010e+01  3.967690e+01
2   1615287600    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
3   1615287660    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
4   1615287660    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
5   1615287720    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
6   1615287720    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
7   1615287780    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
8   1615287780    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
9   1615287840    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
10  1615287840    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
11  1615287900    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
12  1615287900    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
13  1615287960    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
14  1615287960    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.967690e+01  3.967690e+01
15  1615287960    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
16  1615288020    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
17  1615288020    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07
18  1615288080    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
19  1615288080    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.794029e+07  5.794029e+07```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import datetime

# Reuse existing metric and trace DataFrames; ensure numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 0, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 9, 19, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 1) Metric: compute global P95 per (cmdb_id, kpi_name) BEFORE filtering
metric_p95_all = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Filter metric to incident window and IG01 JVM-related KPIs (kpi_name contains 'JVM')
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()
metric_window_ig01_jvm = metric_window[
    (metric_window['cmdb_id'] == 'IG01') &
    (metric_window['kpi_name'].str.contains('JVM', na=False))
].copy()

# Merge with global p95 and detect anomalies (value >= global_p95)
mw = metric_window_ig01_jvm.merge(metric_p95_all, on=['cmdb_id', 'kpi_name'], how='left')
mw['is_anom'] = mw['value'] >= mw['global_p95']

if mw[mw['is_anom']].empty:
    ig01_earliest_anom = "No IG01 JVM-related anomalies found in the window ({} to {}).".format(start_ts, end_ts)
else:
    earliest = mw[mw['is_anom']].sort_values('timestamp', ascending=True).iloc[0]
    ig01_earliest_anom = pd.DataFrame([{
        'earliest_ig01_jvm_anomaly_timestamp': int(earliest['timestamp']),
        'kpi_name': earliest['kpi_name'],
        'value': earliest['value'],
        'global_p95': earliest['global_p95']
    }])

# 2) Trace: compute global P95 per (cmdb_id, trace_name) BEFORE filtering
trace_p95_all = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Filter trace to incident window
trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# Merge and detect anomalies
tw = trace_window.merge(trace_p95_all, on=['cmdb_id', 'trace_name'], how='left')
tw['is_anom'] = tw['value'] >= tw['global_p95']

if tw[tw['is_anom']].empty:
    trace_earliest_anom = "No trace anomalies found in the window ({} to {}).".format(start_ts, end_ts)
else:
    earliest_t = tw[tw['is_anom']].sort_values('timestamp', ascending=True).iloc[0]
    trace_earliest_anom = pd.DataFrame([{
        'earliest_trace_anomaly_timestamp': int(earliest_t['timestamp']),
        'cmdb_id': earliest_t['cmdb_id'],
        'trace_name': earliest_t['trace_name'],
        'value': earliest_t['value'],
        'global_p95': earliest_t['global_p95']
    }])

# Display the two single-row summaries (or messages)
ig01_earliest_anom, trace_earliest_anom

```
Out[6]:
```
Answer:

1) Earliest IG01 JVM anomaly:
- earliest_ig01_jvm_anomaly_timestamp: 1615287600 (2021-03-09 19:00:00 UTC+8)
- kpi_name: JVM-Memory_7778_JVM_Memory_HeapMemoryMax
- value: 2.040136e+09
- global_p95: 2.040136e+09
(Note: value equals the global P95.)

2) Earliest trace anomaly:
- earliest_trace_anomaly_timestamp: 1615287600 (2021-03-09 19:00:00 UTC+8)
- cmdb_id: IG01
- trace_name: trace.to_Tomcat04.duration_mean
- value: 0.46435
- global_p95: 0.46410
(Note: value slightly exceeds the global P95.)

The original code execution output of IPython Kernel is also provided below for reference:

(   earliest_ig01_jvm_anomaly_timestamp                                  kpi_name         value    global_p95
0                           1615287600  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09,    earliest_trace_anomaly_timestamp cmdb_id                       trace_name    value  global_p95
0                        1615287600    IG01  trace.to_Tomcat04.duration_mean  0.46435      0.4641)```
```